In [1]:
import pandas as pd
import heapq
import math
from typing import List

In [2]:
df = pd.read_excel("data/two_jobs_applications.xlsx")
df = df[df["Item"].str.contains("Ability B")]
question_list = list(df.Question)

In [3]:
def get_steps(question):
    num_step = question.count("Step")
    question_step = []
    step_position = []
    for i in range(num_step):
        step_position.append(question.find(f"Step {i+1}"))
        if i > 0:
            question_step.append(question[step_position[i-1]:step_position[i]])
    question_step.append(question[step_position[i]:question.find(f"After completing")])
    question_step = [s.replace("\n", "")[8:] for s in question_step]
    return question_step

def extract_sequence(question: str):
    question_sub = question[38:]
    end_idx = question_sub.find("\n")
    sequence = question_sub[:end_idx].split()
    sequence = [int(s) for s in sequence]
    return sequence

In [4]:
steps_list = [get_steps(q) for q in question_list]
sequence_list = [extract_sequence(question) for question in question_list]

In [5]:
set([s for q in steps_list for s in q])

{'Add all digits that are divisible by 3 to your running total.',
 'Add all even digits to your running total.',
 'Add the count of digits greater than 5 to your running total.',
 'Add the second largest unique digit in the list to your running total.',
 'Add the sum of squares of digits at even indices (0-based) to your running total.',
 'Add the sum of the digits that are prime numbers (2, 3, 5, 7) to your running total.',
 'If there are more even digits than odd digits, multiply your total by 2. Otherwise, add 5.',
 "Multiply your running total by (1 plus the count of digit '7' in the sequence).",
 'Multiply your running total by the product of digits in prime positions.',
 'Subtract all odd digits from your running total.',
 'Subtract the floor of the average of all digits from your running total.',
 'Subtract the sum of the digits in odd positions (1-based) from your running total.'}

In [6]:
def add_div_three(s_list: List[int], running_total: int):
    """
    Add all digits that are divisible by 3 to your running total.
    """
    for n in s_list:
        if n % 3 == 0:
            running_total += n
    return running_total

def add_even_digits(s_list: List[int], running_total: int):
    """
    Add all even digits to your running total.
    """
    for n in s_list:
        if n % 2 == 0:
            running_total += n
    return running_total

def add_count_greater_five(s_list: List[int], running_total: int):
    """
    Add the count of digits greater than 5 to your running total.
    """
    greater_five = [n > 5 for n in s_list]
    return running_total + sum(greater_five)

def add_second_largest_unique(s_list: List[int], running_total: int):
    """
    Add the second largest unique digit in the list to your running total.
    """
    s_list = list(set(s_list))
    second_largest = heapq.nlargest(2, s_list)[1]
    return running_total + second_largest

def add_sum_sq_even_pos(s_list: List[int], running_total: int):
    """
    Add the sum of squares of digits at even indices (0-based) to your running total.
    """
    num_n = len(s_list)
    for i in range(0, len(s_list), 2):
        running_total += s_list[i] ** 2
    return running_total

def add_prime_digits(s_list: List[int], running_total: int):
    """
    Add the sum of the digits that are prime numbers (2, 3, 5, 7) to your running total.
    """
    for n in s_list:
        if n in [2, 3, 5, 7]:
            running_total += n
    return running_total

def if_else_condition(s_list: List[int], running_total: int):
    """
    If there are more even digits than odd digits, multiply your total by 2. Otherwise, add 5.
    """
    num_even = sum([n % 2 == 0 for n in s_list])
    num_odd = sum([n % 2 == 1 for n in s_list])
    if num_even > num_odd:
        return running_total * 2
    else:
        return running_total + 5
    
def mul_count_seven(s_list: List[int], running_total: int):
    """
    Multiply your running total by (1 plus the count of digit '7' in the sequence).
    """
    num_seven = [n == 7 for n in s_list]
    return running_total * (sum(num_seven) + 1)

def mul_prime_pos(s_list: List[int], running_total: int):
    """
    Multiply your running total by the product of digits in prime positions.
    """
    for i in [2, 3, 5, 7]:
        running_total = running_total * s_list[i]
    return running_total

def sub_odd_digits(s_list: List[int], running_total: int):
    """
    Subtract all odd digits from your running total.
    """
    for n in s_list:
        if n % 2 == 1:
            running_total = running_total - n
    return running_total

def sub_floor_avg(s_list: List[int], running_total: int):
    """
    Subtract the floor of the average of all digits from your running total.
    """
    return running_total - int(math.floor(sum(s_list)/len(s_list)))

def sub_odd_pos(s_list: List[int], running_total: int):
    """
    Subtract the sum of the digits in odd positions (1-based) from your running total.
    """
    num_n = len(s_list)
    for i in range(0, len(s_list), 2):
        running_total = running_total - s_list[i]
    return running_total

def step_not_found(s_list: List[int], running_total: int):
    return 0

In [7]:
step_to_func = {
    "Add all digits that are divisible by 3 to your running total.": add_div_three,
    "Add all even digits to your running total.": add_even_digits,
    "Add the count of digits greater than 5 to your running total.": add_count_greater_five,
    "Add the second largest unique digit in the list to your running total.": add_second_largest_unique,
    "Add the sum of squares of digits at even indices (0-based) to your running total.": add_sum_sq_even_pos,
    "Add the sum of the digits that are prime numbers (2, 3, 5, 7) to your running total.": add_prime_digits,
    "If there are more even digits than odd digits, multiply your total by 2. Otherwise, add 5.": if_else_condition,
    "Multiply your running total by (1 plus the count of digit '7' in the sequence).": mul_count_seven,
    "Multiply your running total by the product of digits in prime positions.": mul_prime_pos,
    "Subtract all odd digits from your running total.": sub_odd_digits,
    "Subtract the floor of the average of all digits from your running total.": sub_floor_avg,
    "Subtract the sum of the digits in odd positions (1-based) from your running total.": sub_odd_pos
}

In [8]:
def solve_step(s_list: List[int], steps: List[str]):
    running_total = 0
    for step in steps:
        running_total = step_to_func.get(step, step_not_found)(s_list, running_total)
    return running_total

In [9]:
answer = []
for sequence, steps in zip(sequence_list, steps_list):
    answer.append(solve_step(sequence, steps))

In [10]:
df["Response"] = answer

In [11]:
df.to_csv("intermediate_data/gmaB.csv", index=False)